In [ ]:
- filter by more than 1 column
    - change input to be a dictionary

# Update create_obs_subset

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import h5py
import anndata
from anndata._io.h5ad import read_elem
from scipy.sparse import csr_matrix
from pprint import pprint
import warnings
from tqdm.notebook import tqdm

In [ ]:
filter_column = 'LVL1'
filter_values = [
    'Cardiomyocytes', 
    'Epithelial_cells',
    'Endothelial',
    'Hepatocyte'
    ]
additional_cols_keep = [
    'LVL0',
    'study',
    ]

obs_subset = create_obs_subset(data_dir, filter_column, filter_values, additional_cols_keep)


# update this to return to the user which values aren't present

In [ ]:
def create_obs_subset(
    
    data_dir : str,
    filter_column : str,
    filter_values_obs : list,
    additional_cols_keep : list,
    ):
    
    with h5py.File(data_dir, 'r') as file:
        # Get the index from the 'obs' dataset attributes
        original_index = pd.DataFrame(index=np.vectorize(lambda x: x.decode('utf-8'))(np.array(file["obs"]["_index"], dtype=object)))
        original_index['Original_index_position'] = np.arange(len(original_index))
        
        # Establish the rows to keep
        cols = {}
        col_data = pd.DataFrame(read_elem(file['obs'][filter_column]))
        
        # Filter the DataFrame and check for missing values in one line
        filtered_col_data = col_data[col_data.iloc[:, 0].isin(filter_values_obs)]
        
        # Check if all values in filter_values_obs are present
        missing_values = set(filter_values_obs) - set(filtered_col_data.iloc[:, 0])
    
        if missing_values:
            raise ValueError(f"The following values in filter_values_obs are not present in filter_column: {missing_values}")
            
        col_data.rename(columns={0:filter_column}, inplace=True)
        col_data.index = original_index.iloc[col_data.index].index
        col_data.insert(0, 'Original_index_position', original_index.loc[original_index.index.isin(col_data.index), 'Original_index_position'])
        
        #col_data['orig_index_position'] = original_index.loc[original_index.index.isin(col_data.index), 'orig_index_position']
        cols[filter_column] = col_data
        
        # Get the rest of the columns of interest
        for col in additional_cols_keep:
            col_data = pd.DataFrame(read_elem(file['obs'][col]))
            col_data.rename(columns={0:col}, inplace=True)
            col_data.index = original_index.index
            col_data = col_data[col_data.index.isin(cols[filter_column].index)]
            cols[col] = col_data
            
        out_df = pd.concat(cols.values(), keys=None, axis=1)
        original_col_order = ['Original_index_position'] + list(file['obs'].attrs['column-order']) 
        out_df = out_df.filter(items=original_col_order)
        out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    
    print(f"\033[1mDataFrame output for only columns of interest for cells selected by filtering {filter_column}:\033[0m\n")
    print("Values selected to keep:")
    pprint(filter_values_obs)
    print("")
    display(out_df)
    
    return out_df

# Filter by more than one column

- union or intersect

In [ ]:
filter_information = {
    # filter_column : [filter_value_1, filter_value_2]
    
    'LVL1' : [],
    'LVL2' : [],
    
    
    
}

In [ ]:
def create_obs_subset(
    
    data_dir : str,
    filter_column : str,
    filter_values_obs : list,
    additional_cols_keep : list,
    ):
    
    with h5py.File(data_dir, 'r') as file:
        # Get the index from the 'obs' dataset attributes
        original_index = pd.DataFrame(index=np.vectorize(lambda x: x.decode('utf-8'))(np.array(file["obs"]["_index"], dtype=object)))
        original_index['Original_index_position'] = np.arange(len(original_index))
        
        # Establish the rows to keep
        cols = {}
        col_data = pd.DataFrame(read_elem(file['obs'][filter_column]))
        
        # Filter the DataFrame and check for missing values in one line
        col_data = col_data[col_data.iloc[:, 0].isin(filter_values_obs)]
        
        # Check if all values in filter_values_obs are present
        if not set(filter_values_obs).issubset(set(col_data.iloc[:, 0])):
            raise ValueError("Not all values in filter_values_obs are present in filter_column.")
            
        col_data.rename(columns={0:filter_column}, inplace=True)
        col_data.index = original_index.iloc[col_data.index].index
        col_data.insert(0, 'Original_index_position', original_index.loc[original_index.index.isin(col_data.index), 'Original_index_position'])
        
        #col_data['orig_index_position'] = original_index.loc[original_index.index.isin(col_data.index), 'orig_index_position']
        cols[filter_column] = col_data
        
        # Get the rest of the columns of interest
        for col in additional_cols_keep:
            col_data = pd.DataFrame(read_elem(file['obs'][col]))
            col_data.rename(columns={0:col}, inplace=True)
            col_data.index = original_index.index
            col_data = col_data[col_data.index.isin(cols[filter_column].index)]
            cols[col] = col_data
            
        out_df = pd.concat(cols.values(), keys=None, axis=1)
        original_col_order = ['Original_index_position'] + list(file['obs'].attrs['column-order']) 
        out_df = out_df.filter(items=original_col_order)
        out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    
    print(f"\033[1mDataFrame output for only columns of interest for cells selected by filtering {filter_column}:\033[0m\n")
    print("Values selected to keep:")
    pprint(filter_values_obs)
    print("")
    display(out_df)
    
    return out_df



def check_values_for_key(input_settings, file, filter_key, filter_values):
    
    if input_settings[filter_key]:

        # Check for each value
        not_present_values = [value for value in input_settings[filter_values] if value not in file[filter_key.split('_')[1]].keys()]

        if not_present_values:
            raise ValueError(f"Values not present in {filter_key.split('_')[1]}:", not_present_values)


def check_input_dictionary(input_settings, file):
    # Check all values have been passed in correctly
    required_keys = [
        'data_dir',
        'filter_column_obs',
        'filter_values_obs',
        'additional_cols_keep_obs',
        'filter_column_var',
        'keep_layers',
        'filter_layers',
        'keep_obsm',
        'filter_obsm',
        'keep_obsp',
        'filter_obsp',
        'keep_varm',
        'filter_varm',
        'keep_varp',
        'filter_varp',
        'keep_uns',
        'filter_uns',
    ]
    
    missing_keys = [key for key in required_keys if key not in input_settings]

    if missing_keys:
        raise ValueError(f"Missing required keys: {', '.join(missing_keys)}")

    # Check for values not present in file['obs'].keys()
    obs_cols = [input_settings['filter_column_obs']] + input_settings['additional_cols_keep_obs']
    not_present_values = [value for value in obs_cols if value not in file['obs'].keys()]

    if not_present_values:
        raise ValueError("Values not present in obs columns:", not_present_values)
        
    # Check for values not present in file['var'].keys()   
    not_present_values = [value for value in input_settings['filter_column_var'] if value not in file['var'].keys()]

    if not_present_values:
        raise ValueError("Values not present in var columns:", not_present_values)
           
        
    # Check layers
    check_values_for_key(input_settings, file, 'keep_layers', 'filter_layers')
       
    # Check obsm
    check_values_for_key(input_settings, file, 'keep_obsm', 'filter_obsm')
    
    # Check obsp
    check_values_for_key(input_settings, file, 'keep_obsp', 'filter_obsp')
    
    # Check varm
    check_values_for_key(input_settings, file, 'keep_varm', 'filter_varm')
    
    # Check varp
    check_values_for_key(input_settings, file, 'keep_varp', 'filter_varp')
    
    # Check uns
    check_values_for_key(input_settings, file, 'keep_uns', 'filter_uns')
    
    
    
def grab_row_values(
    
    rows_to_load: list,
    data_dset,
    indices_dset,
    indptr_dset,
    description: str,
    ):
    
    # Initalise empty lists - lists are dynamic in size so adding to a list then converting to a numpy array can be faster then initialising the entire size of the required numpy array
    selected_rows_data = []
    selected_rows_indices = []
    selected_rows_indptr = [0]
    
    # Use tqdm for progress bar to show progression of assigning variables
    for row_idx in tqdm(rows_to_load, desc=f"Processing Rows for {description}", unit="row", position=0, leave=True):
        start_idx = indptr_dset[row_idx]
        end_idx = indptr_dset[row_idx + 1]
        selected_rows_data.extend(data_dset[start_idx:end_idx])
        selected_rows_indices.extend(indices_dset[start_idx:end_idx])
        selected_rows_indptr.append(selected_rows_indptr[-1] + (end_idx - start_idx))
        
    # Convert lists to NumPy arrays
    selected_rows_data = np.array(selected_rows_data)
    selected_rows_indices = np.array(selected_rows_indices)
    selected_rows_indptr = np.array(selected_rows_indptr)
    
    return selected_rows_data, selected_rows_indices, selected_rows_indptr


def create_anndata_subset(input_settings):#, **kwargs): 
    
    # unpack all values
    #for key, value in kwargs.items():
    #    globals()[key] = value
    #kwargs.update(locals())
    
    with h5py.File(input_settings['data_dir'], 'r') as file:
        
        # Check input dictionary
        check_input_dictionary(input_settings, file)
        
        # Get the index from the 'obs' dataset attributes
        original_index = pd.DataFrame(index=np.vectorize(lambda x: x.decode('utf-8'))(np.array(file["obs"]["_index"], dtype=object)))
        original_index['Original_index_position'] = np.arange(len(original_index))
        
        # Establish the rows to keep
        cols = {}
        col_data = pd.DataFrame(read_elem(file['obs'][input_settings['filter_column_obs']]))
        
        # Check that all values in filter_values are in filter_column
        #if not set(filter_values_obs).issubset(set(col_data.iloc[:, 0])):
        
        # Filter the DataFrame and check for missing values in one line
        col_data = col_data[col_data.iloc[:, 0].isin(input_settings['filter_values_obs'])]
        
        # Check if all values in filter_values_obs are present
        if not set(input_settings['filter_values_obs']).issubset(set(col_data.iloc[:, 0])):
            raise ValueError("Not all values in filter_values_obs are present in filter_column_obs.")

        
        
        col_data.rename(columns={0:input_settings['filter_column_obs']}, inplace=True)
        col_data.index = original_index.iloc[col_data.index].index
        col_data.insert(0, 'Original_index_position', original_index.loc[original_index.index.isin(col_data.index), 'Original_index_position'])
        
        #col_data['orig_index_position'] = original_index.loc[original_index.index.isin(col_data.index), 'orig_index_position']
        cols[input_settings['filter_column_obs']] = col_data
        
        # Get the rest of the columns of interest
        for col in input_settings['additional_cols_keep_obs']:
            col_data = pd.DataFrame(read_elem(file['obs'][col]))
            col_data.rename(columns={0:col}, inplace=True)
            col_data.index = original_index.index
            col_data = col_data[col_data.index.isin(cols[input_settings['filter_column_obs']].index)]
            cols[col] = col_data
            
        out_df = pd.concat(cols.values(), keys=None, axis=1)
        original_col_order = ['Original_index_position'] + list(file['obs'].attrs['column-order']) 
        out_df = out_df.filter(items=original_col_order)
        out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    
        # List of row positions to load
        rows_to_load = out_df['Original_index_position'].values
        
        
        # Set warning parameters
        warnings.filterwarnings("ignore", category=FutureWarning)
        
        # Assign variables to query
        data_dset = file['X']['data']
        indices_dset = file['X']['indices']
        indptr_dset = file['X']['indptr']
        
        # Determine the number of columns from the maximum value in the indices array
        num_columns = file["var"][file["var"].attrs["_index"]].shape[0]
        
        selected_rows_data, selected_rows_indices, selected_rows_indptr = grab_row_values(rows_to_load,data_dset,indices_dset,indptr_dset, 'main counts data')
        
        # Create csr_matrix directly from NumPy arrays
        print('Constructing data into csr_matrix format:  \U0001F527', flush=True)
        subset_matrix = csr_matrix(
            (selected_rows_data, selected_rows_indices, selected_rows_indptr),
            shape=(len(rows_to_load), num_columns),
            dtype=file["X"]["data"].dtype
        )
        print('Construction complete \u2705')
        
        if input_settings['filter_column_var']:
                var = pd.DataFrame(read_elem(file['var'][input_settings['filter_column_var']]))
        else:
            var = pd.DataFrame(read_elem(file['var']))

        if input_settings['keep_layers']:
            if not input_settings['filter_layers']:
                layers = {}
                for x in file["layers"].keys():
                    
                    # Assign variables to query
                    data_dset = file['layers'][x]['data']
                    indices_dset = file['layers'][x]['indices']
                    indptr_dset = file['layers'][x]['indptr']
                   
                    name = f'layer {x} data'
                
                    selected_rows_data, selected_rows_indices, selected_rows_indptr = grab_row_values(rows_to_load,data_dset,indices_dset,indptr_dset, name)
                
                    # Create csr_matrix directly from NumPy arrays
                    print('Constructing data into csr_matrix format:  \U0001F527', flush=True)
                    layers[x] = csr_matrix(
                        (selected_rows_data, selected_rows_indices, selected_rows_indptr),
                        shape=(len(rows_to_load), num_columns),
                        dtype=file["layers"][x]["data"].dtype
                    )
                    print('Construction complete \u2705')
                    
            else:
                layers = {}
                for x in (value for value in filter_layers if value in file["layers"].keys()):
                    
                    # Assign variables to query
                    data_dset = file['layers'][x]['data']
                    indices_dset = file['layers'][x]['indices']
                    indptr_dset = file['layers'][x]['indptr']
                    
                    name = f'layer {x} data'
                   
                    selected_rows_data, selected_rows_indices, selected_rows_indptr = grab_row_values(rows_to_load,data_dset,indices_dset,indptr_dset, name)
                
                
                    # Create csr_matrix directly from NumPy arrays
                    print('Constructing data into csr_matrix format:  \U0001F527', flush=True)
                    layers[x] = csr_matrix(
                        (selected_rows_data, selected_rows_indices, selected_rows_indptr),
                        shape=(len(rows_to_load), num_columns),
                        dtype=file["layers"][x]["data"].dtype
                    )
                    print('Construction complete \u2705')
                    
        else:
            layers = None
            
        if input_settings['keep_obsm']:
            if not input_settings['filter_obsm']:
                obsm = {x: anndata._io.h5ad.read_elem(file["obsm"][x])[rows_to_load] for x in file["obsm"].keys()}
            else:
                obsm = {x: anndata._io.h5ad.read_elem(file["obsm"][x])[rows_to_load] for x in filter_obsm if x in file["obsm"].keys()}
        else:
            obsm = None

        if input_settings['keep_obsp']:
            if not input_settings['filter_obsp']:
                obsp = {x: anndata._io.h5ad.read_elem(file["obsp"][x])[rows_to_load][:, rows_to_load]  for x in file["obsp"].keys()}
            else:
                obsp = {x: anndata._io.h5ad.read_elem(file["obsp"][x])[rows_to_load][:, rows_to_load] for x in filter_obsp if x in file["obsp"].keys()}
        else:
            obsp = None

        if input_settings['keep_varm']:
            if not input_settings['filter_varm']:
                varm = anndata._io.h5ad.read_elem(file["varm"])
            else:
                varm = {x: anndata._io.h5ad.read_elem(file["varm"][x]) for x in filter_varm if x in file["varm"].keys()}
        else:
            varm = None


        if input_settings['keep_varp']:
            if not input_settings['filter_varp']:
                varp = anndata._io.h5ad.read_elem(file["varp"])
            else:
                varp = {x: anndata._io.h5ad.read_elem(file["varp"][x]) for x in filter_varp if x in file["varp"].keys()}
        else:
            varp = None


        if input_settings['keep_uns']:
            if not input_settings['filter_uns']:
                uns = anndata._io.h5ad.read_elem(file["uns"])
            else:
                uns = {x: anndata._io.h5ad.read_elem(file["uns"][x]) for x in filter_uns if x in file["uns"].keys()}
        else:
            uns = None
            
        adata = anndata.AnnData(
            X = subset_matrix,
            obs=out_df,
            var=var,
            layers=layers,
            obsm=obsm,
            obsp=obsp,
            varm=varm,
            varp=varp,
            uns=uns,
        )
        warnings.filterwarnings("default")
    
    print('')
    print('')
    print(f"\033[1mSubset anndata object generated successfully\033[0m\n")
    print('\033[1m' + 'Anndata whole preview:' + '\033[0m')
    display(adata)
    print('')
    print('')
    print(f"\033[1mQuick view of the anndata object generated\033[0m\n")
    print(f'Overall shape: {adata.shape}')
    print(f'Min count: {adata.X.min()}')
    print(f'Max count: {adata.X.max()}')
    print('')
    print('\033[1m' + 'obs preview:' + '\033[0m')
    display(adata.obs)
    print('')
    print('\033[1m' + 'var preview:' + '\033[0m')
    display(adata.var)
    print('')
    
    return adata

In [ ]:
# filter by 2 columns
# list of lists for filter_values_obs / dictionary

In [ ]:
import h5py
import pandas as pd
import numpy as np
from anndata._io.h5ad import read_elem

In [ ]:
with h5py.File(input_settings['data_dir'], 'r') as file:
    
    print(file.keys())
    print(file['X'].keys())

In [ ]:
input_settings

In [ ]:
with h5py.File(input_settings['data_dir'], 'r') as file:
    print(file['obs']['_index'])
    original_index = pd.DataFrame(index=np.vectorize(lambda x: x.decode('utf-8'))(np.array(file["obs"]["_index"], dtype=object)))
     #Establish the rows to keep
    cols = {}
    col_data = pd.DataFrame(read_elem(file['obs'][input_settings['filter_column_obs']]))

    # Check that all values in filter_values are in filter_column
    #if not set(filter_values_obs).issubset(set(col_data.iloc[:, 0])):

    # Filter the DataFrame and check for missing values in one line
    col_data = col_data[col_data.iloc[:, 0].isin(input_settings['filter_values_obs'])]
    col_data.rename(columns={0:input_settings['filter_column_obs']}, inplace=True)
    col_data.index = original_index.iloc[col_data.index].index
    col_data.insert(0, 'Original_index_position', original_index.loc[original_index.index.isin(col_data.index), 'Original_index_position'])

    #col_data['orig_index_position'] = original_index.loc[original_index.index.isin(col_data.index), 'orig_index_position']
    cols[input_settings['filter_column_obs']] = col_data

    # Get the rest of the columns of interest
    for col in input_settings['additional_cols_keep_obs']:
        col_data = pd.DataFrame(read_elem(file['obs'][col]))
        col_data.rename(columns={0:col}, inplace=True)
        col_data.index = original_index.index
        col_data = col_data[col_data.index.isin(cols[input_settings['filter_column_obs']].index)]
        cols[col] = col_data

    out_df = pd.concat(cols.values(), keys=None, axis=1)
    original_col_order = ['Original_index_position'] + list(file['obs'].attrs['column-order']) 
    out_df = out_df.filter(items=original_col_order)
    out_df = out_df.loc[:, ~out_df.columns.duplicated()]


In [ ]:
original_index

In [ ]:
read_elem(file['obs'][col])

In [ ]:
col_data

In [ ]:
original_index

In [ ]:
-with pandas once source of inefficiency is the assigninment of memory blocks with non-integrated data expansion/iterative expansions. If the data is not integrated into the saame memory block, you experience massive overheads
which is why large data is usually loaded as arrays or lists.


In [ ]:
dictionary to csr directly - construct dictionary properly as it will already be correct shape etc. 

In [ ]:
add option for parallel 

In [ ]:
with h5py.File(input_settings['data_dir'], 'r') as file:
    print(file['X'].keys())

In [ ]:
with h5py.File(input_settings['data_dir'], 'r') as file:
    print(file['X']['data'])

# ADD A TESTS SECTION!!!!

# USE BELOW FOR IDEAS AND WHAT SHOULD BE INCLUDING

https://www.youtube.com/watch?v=sGw7v-wAQ2s

# test using a generator to perform row by row functions
https://www.youtube.com/watch?v=YsOYMrBNGq8

In [ ]:
from timeit import Timer

def generator(file_name):
    for row in open(file_name, "r"):
        yield row
        
def reader(file_name):
    file = open(file_name, "r")
    result = file.read().split("\n")
    return result

def tester(func):
    csv_data = func("data.csv")
    row_count = 0
    for row in csv_data:
        row_count += 1
        
    return row_count

def main():
    z = Timer("tester(generator)", "from __main__ import tester, generator")
    print("Using generator: %f seconds" % z.timeit(number=5))
    
    y = Timer("tester(reader)", "from __main__ import tester, reader")
    print("Using reader: %f seconds" % y.timeit(number=5))
    
main()